# 6주차 실습 — 텍스트 요약, 텍스트 생성, 그리고 벡터 표현

**자연어처리 | Chapter 6 실습 노트북**

---

### 이번 실습의 목표

강의에서 다룬 세 가지 주제를 **직접 코드로 구현**해 보고, "요약 → 생성 → 표현" 이 하나의 파이프라인으로 이어지는 과정을 눈으로 확인한다.

| PART | 주제 | 대응 절 | 주요 도구 |
|:---:|---|:---:|---|
| 0 | 준비 | — | `numpy`, 내장 코퍼스, 정규표현식 전처리 |
| 1 | 텍스트 요약 | 6.1 | 빈도 → TF-IDF → **TextRank 직접 구현** → ROUGE |
| 2 | 텍스트 생성 | 6.2 | 마르코프 체인(bigram/trigram), temperature · top-k |
| 3 | 벡터 표현 | 6.3 | 원핫 → BoW/TF-IDF → **Word2Vec 직접 구현** → 유추 |

### 진행 방법

1. 아래 **[0] 환경 설정** 셀을 먼저 실행한다. (Colab 기준 1~2분 소요)
2. 위에서부터 순서대로 셀을 실행한다.
3. 노트북 맨 끝의 **[과제]** 4문항(총 100점)이 제출 대상이다.

> **실행 환경** : Google Colab 권장. 로컬 주피터에서도 동일하게 동작한다.
> `scikit-learn`·`nltk`·`networkx`·`gensim` 이 없어도 모든 셀이 동작하도록
> 순수 파이썬/NumPy 대체 구현을 함께 넣어 두었다.

> **[문제] 노트북 사용법**
>
> - `raise NotImplementedError(...)` 로 막혀 있는 부분이 여러분이 채워야 할 곳이다.
>   그 줄을 지우고 직접 코드를 작성한다.
> - 아직 구현하지 않은 과제가 있어도 `런타임 → 모두 실행` 은 끝까지 동작한다.
>   미구현 과제가 있는 셀은 `▶ 미구현 : ...` 이라고만 출력된다.
> - 각 과제 아래의 **채점 셀** 이 통과(`통과!` 출력)해야 정답으로 인정된다.

## [0] 환경 설정

이번 실습에서 사용하는 라이브러리는 다음과 같다.

- **NumPy** — TF-IDF·PageRank·Word2Vec을 직접 구현하는 데 쓰는 유일한 필수 라이브러리 &nbsp;→ *Colab에 이미 설치되어 있다*
- **pandas · matplotlib** — 점수 표와 그래프 출력용 &nbsp;→ *Colab에 이미 설치되어 있다*
- **gensim** — Part 3에서 직접 구현한 Word2Vec과 결과를 비교하기 위한 라이브러리. 설치되어 있지 않으면 비교 셀이 통째로 건너뛰어지므로 **반드시 설치**한다.
- **networkx** — Part 1에서 직접 구현한 PageRank와 결과를 비교하는 데 쓴다.
- **scikit-learn · NLTK** — 있으면 TF-IDF·문장 분리 결과를 대조하고, 없으면 노트북 안의 대체 구현이 그대로 쓰인다.

> **중요** : 이미 설치된 `numpy`/`pandas`/`matplotlib` 을 **다시 설치하면 안 된다.**
> 의존 패키지가 교체되면서 "런타임을 다시 시작하라"는 안내가 뜨고,
> 재시작하면 앞에서 만들어 둔 변수가 모두 사라진다.
> 그래서 아래 셀에서는 **Colab에 없는 `gensim` 과 `networkx` 만** 설치한다.

아래 셀은 한 번만 실행하면 된다.

In [1]:
# ── [0-1] 라이브러리 설치 (로컬 VS Code 기준) ──────────────────────
# 원본은 Colab 기준(!apt-get 으로 나눔 폰트 설치)이라 Windows 에서는 동작하지 않는다.
# 로컬에서는 gensim / networkx 만 있으면 되고, 한글 폰트는 Windows 기본 '맑은 고딕'을 쓴다.
%pip install -q gensim networkx

print("설치 완료")


Note: you may need to restart the kernel to use updated packages.
설치 완료


In [2]:
# ── [0-2] 공통 import 및 한글 폰트 설정 ───────────────────────────
import sys, math, random, re, itertools
from collections import Counter, defaultdict

import numpy as np

# 샘플링·가중치 초기화에 난수를 쓰므로, 매번 같은 결과가 나오도록 시드를 고정한다
random.seed(42)
np.random.seed(42)

# 한글 폰트 설정 ---------------------------------------------------------
# 주의 : apt 로 폰트를 설치해도 matplotlib 의 폰트 캐시는 갱신되지 않기 때문에
#        plt.rcParams['font.family'] 만 바꾸면 그래프의 한글이 그대로 깨진다.
#        addfont() 로 폰트 파일을 직접 등록해야 재시작 없이 즉시 적용된다.
import os
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(NANUM):
    fm.fontManager.addfont(NANUM)
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR']:
        if any(f.name == cand for f in fm.fontManager.ttflist):
            plt.rcParams['font.family'] = cand
            break
plt.rcParams['axes.unicode_minus'] = False

print('matplotlib 한글 폰트 :', plt.rcParams['font.family'])


# 선택 라이브러리 점검 ---------------------------------------------------
def try_import(name):
    """라이브러리를 조용히 import 하고 성공 여부를 반환"""
    try:
        return __import__(name)
    except Exception:
        return None

# 아래 라이브러리는 직접 구현 결과를 대조하는 용도일 뿐, 없어도 실습은 그대로 진행된다
sklearn  = try_import('sklearn')
nltk     = try_import('nltk')
networkx = try_import('networkx')
gensim   = try_import('gensim')

print(f"python   : {sys.version.split()[0]}")
print(f"numpy    : {np.__version__}")
for name, mod in [('scikit-learn', sklearn), ('nltk', nltk),
                  ('networkx', networkx), ('gensim', gensim)]:
    print(f"{name:12s}: {'사용 가능' if mod else '없음 (대체 구현 사용)'}")

# 주피터/코랩이 아닌 환경에서도 display() 가 동작하도록 대비
try:
    display
except NameError:
    def display(x):
        print(x)

matplotlib 한글 폰트 : ['Malgun Gothic']
python   : 3.13.5
numpy    : 2.3.1
scikit-learn: 사용 가능
nltk        : 없음 (대체 구현 사용)
networkx    : 사용 가능
gensim      : 사용 가능


---
# Part 0. 준비

## 0-3. 실습용 코퍼스

외부 다운로드 없이 실습할 수 있도록 짧은 영문 기사 5편과, 각 기사에 대한
**사람이 작성한 참조 요약(reference summary)** 을 노트북 안에 넣어 두었습니다.
참조 요약은 Part 1의 ROUGE 평가에서 정답 역할을 합니다.

In [3]:
# 실습용 코퍼스. 각 기사는 본문(text)과 사람이 작성한 참조 요약(reference)을 함께 가진다
# 참조 요약은 Part 1 의 ROUGE 평가에서 정답 역할을 한다
ARTICLES = [
{
 "id": "A1",
 "title": "Tennis players and the tour schedule",
 "text": (
   "Maria Sharapova has basically no friends as tennis players on the women tour. "
   "The Russian player has no problems in openly speaking about it and in a recent "
   "interview she said she does not really hide many secrets. "
   "I think everyone knows this is my job here. "
   "When I am on the courts or when I am on the court playing I am a competitor and "
   "I want to beat every single person whether they are in the locker room or across "
   "the net. So I am not the one to strike up a conversation about the weather and "
   "know that in the next few minutes I have to go and try to win a tennis match. "
   "I am a pretty competitive girl. I say hello but I am not sending any players "
   "flowers as well. The tour is long and the schedule is demanding, so friendships "
   "are difficult to build and even more difficult to keep. "
   "Sharapova added that when she leaves the court she has a normal life and many "
   "friends away from the courts."
 ),
 "reference": (
   "Sharapova says she has few friends among tennis players because she sees every "
   "opponent as a competitor. She is friendly but keeps her distance, and the long "
   "demanding tour makes friendships hard. Away from the court she has a normal life "
   "with many friends."
 ),
},
{
 "id": "A2",
 "title": "A young player returns from injury",
 "text": (
   "The young player spent almost eight months away from competition after a serious "
   "wrist injury. Doctors told her that surgery was the only option if she wanted to "
   "compete at the highest level again. "
   "Rehabilitation was slow and often frustrating, and for several weeks she could not "
   "hold a racquet at all. She said the hardest part was not the physical pain but the "
   "uncertainty about whether she would ever return to her previous level. "
   "Her coach designed a careful programme that mixed light training with mental "
   "preparation. When she finally came back to the tour she lost in the first round, "
   "but she described the match as one of the happiest moments of her career. "
   "Since then she has climbed steadily in the rankings and reached two finals. "
   "She now says that the injury taught her patience and made her a better competitor."
 ),
 "reference": (
   "After a serious wrist injury and surgery the young player spent eight months in "
   "slow rehabilitation, and the uncertainty was harder than the pain. Her coach mixed "
   "light training with mental preparation. She lost her comeback match but has since "
   "reached two finals and says the injury taught her patience."
 ),
},
{
 "id": "A3",
 "title": "Technology arrives on court",
 "text": (
   "Electronic line calling has changed the way tennis matches are officiated. "
   "For decades line judges stood around the court and their decisions were sometimes "
   "questioned by players and spectators. "
   "The introduction of the challenge system allowed players to ask for a video review "
   "a limited number of times per set, and it quickly became part of the drama of the "
   "sport. Several tournaments have now removed line judges completely and rely on "
   "automated calls made by cameras and software. "
   "Supporters argue that the technology is more accurate and removes an obvious source "
   "of conflict from the game. Critics reply that the human element is part of the "
   "tradition of tennis and that automation reduces the number of jobs around the court. "
   "Players themselves are generally positive because they trust the system and no longer "
   "waste energy arguing about close calls."
 ),
 "reference": (
   "Electronic line calling has replaced human line judges at several tournaments. "
   "The challenge system introduced video review and became part of the drama. "
   "Supporters praise the accuracy while critics miss the human tradition and the lost "
   "jobs, but players are generally positive because they trust the system."
 ),
},
{
 "id": "A4",
 "title": "Why summarization matters for news readers",
 "text": (
   "News organisations produce far more text every day than any reader can follow. "
   "Editors have therefore started to publish short summaries at the top of long articles "
   "so that readers can decide quickly whether to continue. "
   "Automatic summarization systems make this practical at scale because a human editor "
   "cannot write a summary for every single story. "
   "Extractive systems select the most important sentences from the original article and "
   "join them together, which guarantees that the summary contains no invented facts. "
   "Abstractive systems generate new sentences and usually read more naturally, but they "
   "can also produce statements that are not supported by the source. "
   "Most newsrooms therefore combine the two approaches and always keep a human editor in "
   "the loop before publication. Readers report that summaries help them follow more topics "
   "in less time."
 ),
 "reference": (
   "News organisations publish short summaries so readers can decide whether to continue, "
   "and automatic summarization makes this practical at scale. Extractive systems select "
   "original sentences and invent nothing, while abstractive systems read better but may "
   "add unsupported facts. Newsrooms combine both and keep a human editor in the loop."
 ),
},
{
 "id": "A5",
 "title": "Word vectors in practice",
 "text": (
   "Computers cannot work with words directly, so every natural language processing system "
   "must first turn text into numbers. "
   "The simplest approach gives each word its own dimension, which produces very large and "
   "very sparse vectors that say nothing about meaning. "
   "Word embeddings solve this problem by placing each word in a dense space of a few "
   "hundred dimensions, where words that appear in similar contexts end up close together. "
   "The idea rests on the distributional hypothesis, which states that the meaning of a word "
   "is determined by the company it keeps. "
   "Models such as word2vec and GloVe learn these vectors from very large collections of text "
   "without any manual labelling. "
   "Once trained, the vectors support useful operations such as measuring similarity between "
   "words and solving analogy questions. "
   "More recent contextual models produce a different vector for the same word in different "
   "sentences, which finally solves the problem of ambiguous words."
 ),
 "reference": (
   "Natural language processing must turn text into numbers, and one dimension per word gives "
   "huge sparse vectors with no meaning. Word embeddings place words in a dense space where "
   "similar contexts are close, following the distributional hypothesis. Models such as "
   "word2vec and GloVe learn them without labels, and contextual models give different "
   "vectors for ambiguous words."
 ),
},
]

# 원문과 참조 요약의 단어 수를 함께 찍어 요약의 압축 정도를 미리 가늠한다
print(f"기사 {len(ARTICLES)}편")
for a in ARTICLES:
    print(f"  [{a['id']}] {a['title']:45s} 원문 {len(a['text'].split()):4d}단어 / "
          f"참조요약 {len(a['reference'].split()):3d}단어")

기사 5편
  [A1] Tennis players and the tour schedule          원문  177단어 / 참조요약  44단어
  [A2] A young player returns from injury            원문  144단어 / 참조요약  50단어
  [A3] Technology arrives on court                   원문  140단어 / 참조요약  47단어
  [A4] Why summarization matters for news readers    원문  133단어 / 참조요약  49단어
  [A5] Word vectors in practice                      원문  150단어 / 참조요약  56단어


## 0-4. 공통 전처리 함수

`nltk` 가 없어도 동작하도록 **정규표현식 기반 문장 분리기/토크나이저**를 직접 만듭니다.
불용어 목록도 노트북 안에 넣어 두었습니다.

In [4]:
STOPWORDS = set("""
a about above after again against all am an and any are aren't as at be because been
before being below between both but by can cannot could couldn't did didn't do does
doesn't doing don't down during each few for from further had hadn't has hasn't have
haven't having he her here hers herself him himself his how i if in into is isn't it
its itself let's me more most mustn't my myself no nor not of off on once only or other
ought our ours ourselves out over own same shan't she should shouldn't so some such than
that the their theirs them themselves then there these they this those through to too
under until up very was wasn't we were weren't what when where which while who whom why
with won't would wouldn't you your yours yourself yourselves
""".split())

SENT_END = re.compile(r'(?<=[.!?])\s+(?=[A-Z"\'])')
WORD_RE = re.compile(r"[a-zA-Z]+")

def split_sentences(text):
    """마침표/물음표/느낌표 뒤에서 문장을 나눔 (nltk 가 있으면 nltk 사용)"""
    if nltk is not None:
        try:
            from nltk.tokenize import sent_tokenize
            return [s.strip() for s in sent_tokenize(text) if s.strip()]
        except Exception:
            pass  # punkt 미설치 등 -> 아래 정규식으로 대체
    return [s.strip() for s in SENT_END.split(text.strip()) if s.strip()]

def tokenize(text):
    """소문자 알파벳 단어만 남김"""
    return WORD_RE.findall(text.lower())

def preprocess(sentence, remove_stop=True):
    """문장 -> 정제된 단어 리스트"""
    words = tokenize(sentence)
    if remove_stop:
        words = [w for w in words if w not in STOPWORDS and len(w) > 1]
    return words

# 동작 확인
demo = ARTICLES[0]['text']
sents = split_sentences(demo)
print(f"문장 수 : {len(sents)}")
print("첫 문장 :", sents[0])
print("토큰    :", preprocess(sents[0])[:12])

문장 수 : 9
첫 문장 : Maria Sharapova has basically no friends as tennis players on the women tour.
토큰    : ['maria', 'sharapova', 'basically', 'friends', 'tennis', 'players', 'women', 'tour']


---
# Part 1. 텍스트 요약

## 1-1. 방법 1 — 단어 빈도 기반 요약

가장 단순한 추출 요약입니다.

1. 불용어를 제거한 뒤 문서 전체의 단어 빈도를 셈
2. 문장 점수 = 문장에 포함된 단어들의 빈도 합 ÷ 문장 길이(정규화)
3. 점수 상위 n개 문장을 **원문 순서대로** 이어 붙임

> 3번이 중요합니다. 점수 순서로 붙이면 읽기 어려운 요약이 됩니다.

In [6]:
def summarize_by_frequency(text, n=3): 
    #여기서 n=e은 요약으로 뽑을 문장의 개수이다. 이코드는 unigram기반
    sentences = split_sentences(text)
    if len(sentences) <= n:
        return ' '.join(sentences), list(range(len(sentences)))

    # 1) 문서 전체 단어 빈도
    freq = Counter()
    for s in sentences:
        freq.update(preprocess(s))
    if not freq:
        return ' '.join(sentences[:n]), list(range(n))
    max_f = max(freq.values())
    freq = {w: c / max_f for w, c in freq.items()}   
    #가장 많이 나온 단어의 횟수(max_f)를 기준으로 모든 횟수를 나눠 0~1 범위로 정규화한다
    #이값이 곧 기사에서 이 단어가 얼마나 중요한지를 의미하는 점수가 됨

    # 2) 문장 점수
    scores = []
    for s in sentences:
        words = preprocess(s)
        score = sum(freq.get(w, 0.0) for w in words) / (len(words) + 1e-9)
        scores.append(score)
    #문장 안 단어들의 중요도를 모두 더한 뒤 문장의 단어수(len(words))로 나눈다
    #문장에 포함된 단어 중요도의 평균을 구하는 건데, 길이로 나누는 이유는 합만 사용하면 단어가 많은 긴문장이 유리하므로
    
    
    # 3) 상위 n개를 원문 순서대로
    top_idx = sorted(sorted(range(len(scores)), key=lambda i: -scores[i])[:n])
    return ' '.join(sentences[i] for i in top_idx), top_idx
    #sorted가 두 번 나오면 안쪽부터 읽으면 됨
    #문장 번호들을 점수내린 순으로 내림차순 정렬한다 ('-'붙이는게 내림차순 설정)
    #그중 [:n] 앞의 n개만 남긴다 (최상단에서 n=3으로 설정)

art = ARTICLES[0]
summary, idx = summarize_by_frequency(art['text'], n=3)
print(f"[{art['id']}] 빈도 기반 요약 (선택 문장 {idx})\n")
print(summary)

[A1] 빈도 기반 요약 (선택 문장 [0, 7, 8])

Maria Sharapova has basically no friends as tennis players on the women tour. The tour is long and the schedule is demanding, so friendships are difficult to build and even more difficult to keep. Sharapova added that when she leaves the court she has a normal life and many friends away from the courts.


## 1-2. 방법 2 — TF-IDF 기반 요약

빈도만 쓰면 그 문서에서 흔하다는 이유만으로 점수가 올라갑니다.
TF-IDF는 **다른 문서에는 잘 안 나오는 단어**에 높은 가중치를 줍니다.

$$\text{TF-IDF}(t,d) = \text{TF}(t,d) \times \left[\log\frac{N}{1+\text{df}(t)} + 1\right]$$

여기서는 *문장 하나를 하나의 문서*로 보고 계산합니다.

In [ ]:
def compute_tfidf(docs_tokens):
    """docs_tokens: 문서(=문장)별 토큰 리스트 -> (tfidf 행렬, 어휘 리스트)"""
    vocab = sorted({w for doc in docs_tokens for w in doc})
    vidx = {w: i for i, w in enumerate(vocab)}
    N, V = len(docs_tokens), len(vocab)
    #모든 문장에 나온 단어를 중복없이 모아 정렬해 vocab목록을 만든다
    #vidx 사전으로 단어를 열 번호로 바꿔주고
    #N은 문장수(docs_token), V는 어휘수로 설정한다. N X V 행렬이 된다

    tf = np.zeros((N, V))
    for i, doc in enumerate(docs_tokens):
        if not doc:
            continue
        for w, c in Counter(doc).items():
            tf[i, vidx[w]] = c / len(doc)
    #0으로 채운 9 X V 행렬을 먼저 만들고
    #각 문장에서 단어가 몇번나왔는지 센 다음(c) 그걸 문장길이로 나눈값을 구한다
    #불용어만 있어서 단어가 하나도 안남으면 문장을 건너뛰어 오류 방지 (if not doc: continue)

    df = (tf > 0).sum(axis=0)
    # +1 : df 가 커지면 log 값이 음수가 되어 점수를 깎아내리므로 이를 방지하는 보정항이다.
    idf = np.log(N / (1 + df)) + 1.0
    return tf * idf, vocab
    #IDF절차인데, 여러 문장에 흔하게 나오는 단어일수록 점수를 낮춘다
    #그 다음에는 tf*idf 진행

def summarize_by_tfidf(text, n=3):
    sentences = split_sentences(text)
    if len(sentences) <= n:
        return ' '.join(sentences), list(range(len(sentences)))

    tokens = [preprocess(s) for s in sentences]
    tfidf, vocab = compute_tfidf(tokens)
    scores = tfidf.sum(axis=1)              # 문장별 TF-IDF 합

    top_idx = sorted(sorted(range(len(scores)), key=lambda i: -scores[i])[:n])
    return ' '.join(sentences[i] for i in top_idx), top_idx
#문장 점수를 매기고 고른다. 점수 상위 3개를 고른 뒤 '원문 순서를 참고하여 다시 정렬'한다

summary_tfidf, idx_tfidf = summarize_by_tfidf(art['text'], n=3)
print(f"[{art['id']}] TF-IDF 기반 요약 (선택 문장 {idx_tfidf})\n")
print(summary_tfidf)

[A1] TF-IDF 기반 요약 (선택 문장 [1, 2, 5])

The Russian player has no problems in openly speaking about it and in a recent interview she said she does not really hide many secrets. I think everyone knows this is my job here. I am a pretty competitive girl.


In [8]:
# scikit-learn 이 있으면 같은 결과가 나오는지 비교해 봅니다.
if sklearn is not None:
    from sklearn.feature_extraction.text import TfidfVectorizer
    sentences = split_sentences(art['text'])
    X = TfidfVectorizer(stop_words='english').fit_transform(sentences)
    sk_scores = np.asarray(X.sum(axis=1)).ravel()
    sk_idx = sorted(sorted(range(len(sk_scores)), key=lambda i: -sk_scores[i])[:3])
    print("직접 구현 선택 문장 :", idx_tfidf)
    print("scikit-learn 선택 문장:", sk_idx)
    print("\n(정규화 방식이 조금 달라 완전히 같지 않을 수 있습니다)")
else:
    print("scikit-learn 이 없어 비교를 건너뜁니다.")

직접 구현 선택 문장 : [1, 2, 5]
scikit-learn 선택 문장: [1, 3, 4]

(정규화 방식이 조금 달라 완전히 같지 않을 수 있습니다)


정리하면, TF-IDF는 원래 "문서끼리 구별되는 단어 찾기"에 좋은 도구임. 그런데 문장 하나를 문서로 놓고 그대로 요약에 쓰니, 기사 주제를 대표하는 문장이 아니라 엉뚱하게 튀는 문장을 고르게 된 것이다.

## 1-3. 방법 3 — TextRank 직접 구현

TextRank는 **문장을 노드, 문장 간 유사도를 간선 가중치**로 하는 그래프에 PageRank를 적용합니다.

$$WS(V_i) = \frac{1-d}{n} + d \sum_{j \in In(V_i)} \frac{w_{ji}}{\sum_{k \in Out(V_j)} w_{jk}} WS(V_j)$$

### 단계
1. 문장 분리 → 전처리
2. 문장을 벡터로 변환 (여기서는 TF-IDF 벡터를 사용 — GloVe가 없어도 동작)
3. 코사인 유사도로 n×n 유사도 행렬 계산
4. 행렬을 전이 행렬로 정규화하고 **멱승법(power iteration)** 으로 PageRank 계산
5. 점수 상위 n개 문장 선택

In [ ]:
def cosine_sim_matrix(vectors):
    """행 단위 코사인 유사도 행렬 (대각선은 0)"""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-9
    unit = vectors / norms
    sim = unit @ unit.T
    np.fill_diagonal(sim, 0.0)
    return np.clip(sim, 0.0, None)          # 음수 유사도는 0으로


def pagerank(sim_matrix, d=0.85, max_iter=200, tol=1e-8):
    """가중 그래프에 대한 PageRank (멱승법)"""
    n = sim_matrix.shape[0]
    if n == 0:
        return np.array([])
    # 열 기준 전이 행렬로 정규화
    # 유사도가 모두 0 인 고립 문장(dangling node)을 표시해 둔다.
    # 처리하지 않으면 매 반복마다 확률 질량이 새어 나가 점수의 합이 1 이 되지 않는다.
    is_dangling = (sim_matrix.sum(axis=1) == 0)
    col_sum  = sim_matrix.sum(axis=1, keepdims=True)
    col_sum[col_sum == 0] = 1e-9
    M = (sim_matrix / col_sum).T

    scores = np.ones(n) / n
    for it in range(max_iter):
        # 고립 문장이 가진 확률 질량은 모든 노드에 균등 재분배한다 (networkx 구현과 동일).
        new  = (1 - d) / n + d * (M @ scores) + d * scores[is_dangling].sum() / n
        if np.abs(new - scores).sum() < tol:
            scores = new
            break
        scores = new
    return scores


def textrank_summarize(text, n=3, d=0.85, return_detail=False):
    sentences = split_sentences(text)
    if len(sentences) <= n:
        out = ' '.join(sentences)
        return (out, list(range(len(sentences))), None) if return_detail else (out, list(range(len(sentences))))

    tokens = [preprocess(s) for s in sentences]
    vectors, _ = compute_tfidf(tokens)          # 2) 문장 벡터
    sim = cosine_sim_matrix(vectors)            # 3) 유사도 행렬
    scores = pagerank(sim, d=d)                 # 4) PageRank

    top_idx = sorted(sorted(range(len(scores)), key=lambda i: -scores[i])[:n])
    summary = ' '.join(sentences[i] for i in top_idx)
    if return_detail:
        return summary, top_idx, {'sentences': sentences, 'sim': sim, 'scores': scores}
    return summary, top_idx


summary_tr, idx_tr, detail = textrank_summarize(art['text'], n=3, return_detail=True)
print(f"[{art['id']}] TextRank 요약 (선택 문장 {idx_tr})\n")
print(summary_tr)

In [ ]:
# 문장별 TextRank 점수 확인
sc = detail['scores']
order = np.argsort(-sc)
print(f"{'순위':<4}{'문장#':<6}{'점수':<10}문장 앞부분")
print('-' * 90)
for rank, i in enumerate(order, 1):
    print(f"{rank:<4}{i:<6}{sc[i]:<10.5f}{detail['sentences'][i][:60]}...")

In [ ]:
# 유사도 행렬 시각화
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(detail['sim'], cmap='Blues')
ax.set_title('Sentence Similarity Matrix')
ax.set_xlabel('sentence index'); ax.set_ylabel('sentence index')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# networkx 가 있으면 라이브러리 PageRank 와 비교
if networkx is not None:
    import networkx as nx
    g = nx.from_numpy_array(detail['sim'])
    nx_scores = nx.pagerank(g, alpha=0.85)
    nx_scores = np.array([nx_scores[i] for i in range(len(detail['sentences']))])
    mine = detail['scores']
    print(f"{'문장#':<6}{'직접 구현':<14}{'networkx':<14}차이")
    for i in range(len(mine)):
        print(f"{i:<6}{mine[i]:<14.6f}{nx_scores[i]:<14.6f}{abs(mine[i]-nx_scores[i]):.2e}")
    print("\n상위 3문장 (직접):", sorted(np.argsort(-mine)[:3]))
    print("상위 3문장 (networkx):", sorted(np.argsort(-nx_scores)[:3]))
else:
    print("networkx 가 없어 비교를 건너뜁니다. (직접 구현 결과만 사용)")

## 1-4. 세 가지 방법 비교

같은 기사에 세 가지 요약기를 모두 적용해 결과를 나란히 봅니다.

In [ ]:
# 같은 기사에 세 요약기를 모두 적용해 선택되는 문장이 어떻게 달라지는지 비교한다
def compare_summaries(article, n=3):
    print("=" * 92)
    print(f"[{article['id']}] {article['title']}")
    print("=" * 92)
    for name, fn in [('빈도 기반  ', summarize_by_frequency),
                     ('TF-IDF     ', summarize_by_tfidf),
                     ('TextRank   ', textrank_summarize)]:
        # 세 함수 모두 (요약문, 선택된 문장 인덱스, ...) 형태로 반환하므로 앞의 두 개만 받는다
        s, idx = fn(article['text'], n)[:2]
        print(f"\n▶ {name} (문장 {idx})")
        print("  " + s)
    print(f"\n▶ 참조 요약 (사람)")
    print("  " + article['reference'])

compare_summaries(ARTICLES[0])

In [ ]:
compare_summaries(ARTICLES[3])

## 1-5. 요약 평가 — ROUGE 직접 구현

$$\text{ROUGE-N}_{recall}=\frac{|\text{겹치는 n-gram}|}{|\text{참조 요약의 n-gram}|},\quad
\text{ROUGE-N}_{precision}=\frac{|\text{겹치는 n-gram}|}{|\text{생성 요약의 n-gram}|}$$

**ROUGE-L** 은 최장 공통 부분 수열(LCS) 길이를 사용합니다. 어순을 부분적으로 반영합니다.

In [ ]:
def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def rouge_n(reference, candidate, n=1):
    ref, cand = tokenize(reference), tokenize(candidate)
    ref_ng, cand_ng = ngrams(ref, n), ngrams(cand, n)
    overlap = sum((ref_ng & cand_ng).values())      # 클리핑된 중복 개수
    r = overlap / max(sum(ref_ng.values()), 1)
    p = overlap / max(sum(cand_ng.values()), 1)
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {'precision': p, 'recall': r, 'f1': f}


def lcs_length(a, b):
    """동적 계획법으로 최장 공통 부분 수열 길이"""
    m, k = len(a), len(b)
    dp = [[0] * (k + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, k + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][k]


def rouge_l(reference, candidate):
    ref, cand = tokenize(reference), tokenize(candidate)
    l = lcs_length(ref, cand)
    r = l / max(len(ref), 1)
    p = l / max(len(cand), 1)
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {'precision': p, 'recall': r, 'f1': f}


def rouge_all(reference, candidate):
    return {'ROUGE-1': rouge_n(reference, candidate, 1),
            'ROUGE-2': rouge_n(reference, candidate, 2),
            'ROUGE-L': rouge_l(reference, candidate)}


# 강의 슬라이드의 검증 예제
ref_demo  = "the cat was found under the bed"
cand_demo = "the cat was under the bed"
r1 = rouge_n(ref_demo, cand_demo, 1)
print("검증 예제 (슬라이드와 동일해야 함)")
print(f"  ROUGE-1 Recall    = {r1['recall']:.3f}   (기대값 0.857)")
print(f"  ROUGE-1 Precision = {r1['precision']:.3f}   (기대값 1.000)")
print(f"  ROUGE-1 F1        = {r1['f1']:.3f}   (기대값 0.923)")

In [ ]:
def evaluate_all(n=3):
    """기사 5편 x 요약기 3종에 대해 ROUGE 점수 표를 만듦"""
    methods = {'빈도':     summarize_by_frequency,
               'TF-IDF':   summarize_by_tfidf,
               'TextRank': textrank_summarize}
    rows = []
    for a in ARTICLES:
        for mname, fn in methods.items():
            cand = fn(a['text'], n)[0]
            sc = rouge_all(a['reference'], cand)
            rows.append({'기사': a['id'], '방법': mname,
                         'R1_F1': sc['ROUGE-1']['f1'],
                         'R2_F1': sc['ROUGE-2']['f1'],
                         'RL_F1': sc['ROUGE-L']['f1']})
    return rows

rows = evaluate_all(n=3)

# pandas 가 있으면 표로, 없으면 텍스트로 출력
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    print("● 기사별 점수\n")
    print(df.pivot(index='기사', columns='방법',
                   values='R1_F1').round(4).to_string())
    print("\n● 방법별 평균 (F1)\n")
    print(df.groupby('방법')[['R1_F1', 'R2_F1', 'RL_F1']].mean().round(4).to_string())
except ImportError:
    for r in rows:
        print(r)

### 결과를 어떻게 읽어야 할까?

- ROUGE 점수는 **절대적인 품질**이 아니라 *참조 요약과 얼마나 겹치는가*를 나타냅니다.
- 추출 요약은 원문 문장을 그대로 쓰므로 사람이 다시 쓴 참조 요약과는 표현이 달라
  점수가 낮게 나오는 경우가 많습니다.
- 따라서 **같은 조건에서 방법끼리 비교**하는 용도로 쓰는 것이 안전합니다.

## 1-6. (선택) 사전 학습 모델로 추상 요약

인터넷과 `transformers` 라이브러리를 쓸 수 있는 환경이라면 아래 셀을 실행해
추출 요약과 생성 요약의 차이를 직접 확인해 보세요. (실행에 몇 분이 걸릴 수 있습니다)

In [ ]:
RUN_TRANSFORMERS = False   # True 로 바꾸면 실행됩니다

if RUN_TRANSFORMERS:
    try:
        from transformers import pipeline
        summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
        out = summarizer(ARTICLES[0]['text'], max_length=90, min_length=35,
                         do_sample=False)[0]['summary_text']
        print("▶ 추상 요약 (distilbart)\n ", out)
        print("\n▶ ROUGE vs 참조 요약")
        for k, v in rouge_all(ARTICLES[0]['reference'], out).items():
            print(f"  {k}: F1={v['f1']:.4f}")
    except Exception as e:
        print("실행 실패:", e)
else:
    print("RUN_TRANSFORMERS = True 로 바꾸면 사전 학습 요약 모델을 사용합니다.")

---
# Part 2. 텍스트 생성

## 2-1. 마르코프 체인 (bigram)

마르코프 가정 : $P(w_t \mid w_1,\dots,w_{t-1}) \approx P(w_t \mid w_{t-1})$

즉 **직전 단어 하나만 보고** 다음 단어를 고릅니다.
먼저 코퍼스에서 연속한 단어 쌍을 모아 전이 사전을 만듭니다.

In [ ]:
# 생성 실습용 코퍼스: 기사 5편의 본문을 모두 이어 붙임 (문장부호 유지)
GEN_CORPUS_TEXT = ' '.join(a['text'] for a in ARTICLES)
GEN_TOKENS = re.findall(r"[A-Za-z']+|[.,!?]", GEN_CORPUS_TEXT.lower())

print(f"토큰 수 : {len(GEN_TOKENS)},  고유 토큰 수 : {len(set(GEN_TOKENS))}")
print("앞 20개 :", GEN_TOKENS[:20])

In [ ]:
def build_bigram_model(tokens):
    """{현재 단어: Counter(다음 단어 -> 빈도)}"""
    model = defaultdict(Counter)
    for w1, w2 in zip(tokens, tokens[1:]):
        model[w1][w2] += 1
    return model

bigram_model = build_bigram_model(GEN_TOKENS)

# 전이 확률 확인
def show_transitions(model, word, k=5):
    counter = model.get(word)
    if not counter:
        print(f"'{word}' 다음에 오는 단어 정보가 없습니다."); return
    total = sum(counter.values())
    print(f"P(다음 단어 | '{word}')  — 상위 {k}개")
    for w, c in counter.most_common(k):
        print(f"   {w:<15s} {c:>3d}/{total:<3d} = {c/total:.3f}")

show_transitions(bigram_model, 'the')
print()
show_transitions(bigram_model, 'tennis')

## 2-2. 확률에 따라 다음 단어 고르기 — temperature 와 top-k

세 가지 디코딩 전략을 한 함수로 구현합니다.

| 전략 | 설명 |
|---|---|
| greedy | 가장 확률이 높은 단어 하나를 선택 (`greedy=True`) |
| temperature | 확률을 $p^{1/T}$ 로 변형. $T<1$ 보수적, $T>1$ 다양 |
| top-k | 확률 상위 k개 후보 안에서만 샘플링 |

In [ ]:
def sample_next(counter, temperature=1.0, top_k=None, greedy=False, rng=None):
    """전이 Counter 에서 다음 단어를 선택.

    rng 를 넘기면 그 난수 생성기만 사용한다. np.random.seed() 로 전역 난수를
    건드리면 이 함수를 부른 뒤 실행되는 다른 셀(Word2Vec 학습 등)의 결과까지
    바뀌어 버리므로, 시드는 반드시 지역 난수 생성기에 가둔다.
    """
    if rng is None:
        rng = np.random.default_rng()

    words = list(counter.keys())
    counts = np.array([counter[w] for w in words], dtype=float)
    probs = counts / counts.sum()

    if greedy:
        return words[int(np.argmax(probs))]

    if top_k is not None and top_k < len(words):
        keep = np.argsort(-probs)[:top_k]
        words = [words[i] for i in keep]
        probs = probs[keep]
        probs = probs / probs.sum()

    if temperature != 1.0:
        probs = probs ** (1.0 / max(temperature, 1e-6))
        probs = probs / probs.sum()

    return words[int(rng.choice(len(words), p=probs))]


def _pick(rng, seq):
    """지역 난수 생성기로 리스트에서 하나를 고른다 (random.choice 의 전역 난수 회피)"""
    seq = list(seq)
    return seq[int(rng.integers(len(seq)))]


def generate_text(model, n_words=40, start=None, temperature=1.0,
                  top_k=None, greedy=False, seed=None):
    """bigram 전용 생성기 — 모델의 키가 '문자열' 이어야 한다.

    build_ngram_model() 이 만든 모델은 키가 튜플이라 여기에 넣으면
    "'tuple' object has no attribute 'isalpha'" 오류가 났었다.
    원인을 알기 어려운 오류 대신, 무엇을 써야 하는지 알려 주고 멈춘다.
    """
    if model and isinstance(next(iter(model)), tuple):
        raise TypeError(
            "generate_text() 는 키가 문자열인 bigram 모델(build_bigram_model) 전용이다. "
            "build_ngram_model() 이 만든 모델은 상태가 튜플이므로 "
            "generate_ngram(model, n=...) 을 사용할 것.")

    rng = np.random.default_rng(seed)      # 시드는 이 함수 안에만 영향을 준다
    word = start if start in model else _pick(rng, [w for w in model if w.isalpha()])
    out = [word]
    for _ in range(n_words - 1):
        if word not in model or not model[word]:
            word = _pick(rng, model.keys())
        word = sample_next(model[word], temperature, top_k, greedy, rng=rng)
        out.append(word)
    text = ' '.join(out)
    return re.sub(r"\s+([.,!?])", r"\1", text)   # 문장부호 앞 공백 제거


print("▶ greedy (항상 최빈 단어)")
print(" ", generate_text(bigram_model, 40, start='the', greedy=True), "\n")

for T in [0.5, 1.0, 1.5]:
    print(f"▶ temperature = {T}")
    print(" ", generate_text(bigram_model, 40, start='the', temperature=T, seed=7), "\n")

print("▶ top-k = 3, temperature = 1.0")
print(" ", generate_text(bigram_model, 40, start='the', top_k=3, seed=7))

**관찰 포인트**

- `greedy` 는 곧바로 같은 구절을 반복합니다 (마르코프 체인의 대표적 실패 사례).
- `temperature` 를 낮추면 안전하지만 단조롭고, 높이면 다양하지만 문법이 무너집니다.
- `top_k` 는 꼬리 확률의 이상한 단어를 잘라내므로 반복과 다양성의 절충이 됩니다.

## 2-3. trigram 으로 확장하기

문맥을 두 단어로 늘리면 문장이 훨씬 자연스러워집니다.
상태를 `(w1, w2)` 튜플로 두는 것이 전부입니다.

In [ ]:
# bigram 모델을 임의의 n 으로 일반화하고, bigram 과 trigram 의 생성 품질을 비교한다
def build_ngram_model(tokens, n=3):
    """n-gram 모델: {(w1,...,w_{n-1}): Counter(다음 단어)}  ← 상태(키)가 '튜플' 이다"""
    model = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        state = tuple(tokens[i:i+n-1])      # 직전 n-1개 단어가 상태(문맥)가 된다
        model[state][tokens[i+n-1]] += 1    # 그 상태 다음에 등장한 단어의 횟수를 센다
    return model


def generate_ngram(model, n=3, n_words=40, temperature=1.0, top_k=None, seed=None):
    """상태가 튜플인 n-gram 모델용 생성기 (n=2 도 그대로 쓸 수 있다)"""
    rng = np.random.default_rng(seed)       # 전역 난수를 건드리지 않는다
    # 시작 상태는 문장부호가 섞이지 않은 것 중에서 골라야 문장이 자연스럽게 시작된다
    state = _pick(rng, [s for s in model if all(w.isalpha() for w in s)])
    out = list(state)
    for _ in range(n_words - (n - 1)):
        # 학습 데이터에 없는 상태에 도달하면 더 진행할 수 없으므로 임의의 상태에서 다시 시작한다
        if state not in model or not model[state]:
            state = _pick(rng, model.keys())
        nxt = sample_next(model[state], temperature, top_k, rng=rng)
        out.append(nxt)
        state = tuple(out[-(n-1):])           # 방금 생성한 단어를 포함하도록 상태를 한 칸 옮긴다
    return re.sub(r"\s+([.,!?])", r"\1", ' '.join(out))


trigram_model = build_ngram_model(GEN_TOKENS, n=3)
print(f"bigram  상태 수 : {len(bigram_model)}")
print(f"trigram 상태 수 : {len(trigram_model)}\n")

print("▶ bigram (문자열 상태 모델 → generate_text)")
print(" ", generate_text(bigram_model, 40, temperature=1.0, seed=3), "\n")
print("▶ trigram (튜플 상태 모델 → generate_ngram)")
print(" ", generate_ngram(trigram_model, n=3, n_words=40, temperature=1.0, seed=3))

### n이 커질 때의 문제 — 데이터 희소성

`n` 을 키우면 문맥은 좋아지지만, 각 상태의 후보가 1개뿐이 되어
결국 **원문을 그대로 외워서 뱉는** 현상이 나타납니다. 직접 확인해 봅시다.

In [ ]:
# n 을 키울수록 각 상태의 다음 단어 후보가 하나로 고정되는 비율(데이터 희소성)을 측정한다
print(f"{'n':<4}{'상태 수':<10}{'후보가 1개뿐인 상태 비율':<26}")
print('-' * 45)
for n in [2, 3, 4, 5, 6]:
    m = build_ngram_model(GEN_TOKENS, n=n)
    # 후보가 1개뿐인 상태 = 선택의 여지가 없어 원문을 그대로 재생하게 되는 상태
    single = sum(1 for c in m.values() if len(c) == 1)
    print(f"{n:<4}{len(m):<10}{single/len(m):.1%}")

## 2-4. (선택) 신경망 언어 모델과 비교

`transformers` 를 사용할 수 있다면 GPT-2로 같은 시작 문구를 이어 써 보고
마르코프 체인 결과와 비교해 보세요.

In [ ]:
RUN_GPT2 = False   # True 로 바꾸면 실행됩니다

if RUN_GPT2:
    try:
        from transformers import pipeline, set_seed
        gen = pipeline('text-generation', model='gpt2')
        set_seed(42)
        out = gen("Electronic line calling has changed the way tennis",
                  max_length=60, num_return_sequences=1)[0]['generated_text']
        print("▶ GPT-2\n ", out)
    except Exception as e:
        print("실행 실패:", e)
else:
    print("RUN_GPT2 = True 로 바꾸면 GPT-2 생성 결과를 볼 수 있습니다.")

---
# Part 3. 벡터 표현

## 3-1. 원핫 인코딩과 그 한계

어휘 크기 $N$ 에 대해 각 단어는 길이 $N$ 의 벡터가 되고, 자기 인덱스만 1입니다.

In [ ]:
# 원핫 인코딩이 단어 사이의 의미적 거리를 전혀 담지 못한다는 점을 코사인 유사도로 확인한다
vocab_demo = ['나는', '밥을', '빵을', '먹었다']
N = len(vocab_demo)
onehot = np.eye(N, dtype=int)   # 단위행렬의 i번째 행이 곧 i번 단어의 원핫 벡터

for w, v in zip(vocab_demo, onehot):
    print(f"{w:<6s} -> {v}")

def cos(a, b):
    # 코사인 유사도. 영벡터가 들어오면 0으로 나누게 되므로 그때는 0.0 을 돌려준다
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else 0.0

print(f"\ncos(밥을, 빵을)  = {cos(onehot[1], onehot[2]):.3f}")
print(f"cos(밥을, 먹었다) = {cos(onehot[1], onehot[3]):.3f}")
print("\n→ 서로 다른 단어는 무조건 유사도 0. 의미의 가까움을 전혀 표현하지 못합니다.")

In [ ]:
# 실제 코퍼스로 원핫 인코딩을 만들면 얼마나 커지는가?
corpus_vocab = sorted(set(tokenize(GEN_CORPUS_TEXT)))
V = len(corpus_vocab)
n_tokens = len(tokenize(GEN_CORPUS_TEXT))
print(f"어휘 크기 V = {V}")
print(f"문서 전체를 원핫 행렬로 만들면 : {n_tokens} x {V} = {n_tokens*V:,} 개 값")
print(f"그중 0이 아닌 값 : {n_tokens:,} 개  (희소도 {1 - 1/V:.4%})")
print(f"float32 로 저장 시 메모리 : {n_tokens*V*4/1024/1024:.1f} MB")
print(f"\n같은 정보를 100차원 임베딩으로 : {n_tokens*100*4/1024/1024:.2f} MB")

## 3-2. BoW 와 TF-IDF 로 문서 표현하기

문서(기사) 5편을 벡터로 만들고 서로의 유사도를 비교합니다.

In [ ]:
docs = [a['text'] for a in ARTICLES]
doc_ids = [a['id'] for a in ARTICLES]
doc_tokens = [preprocess(d) for d in docs]

# --- BoW ---
vocab = sorted({w for t in doc_tokens for w in t})
vidx = {w: i for i, w in enumerate(vocab)}
bow = np.zeros((len(docs), len(vocab)))
for i, t in enumerate(doc_tokens):
    for w, c in Counter(t).items():
        bow[i, vidx[w]] = c

# --- TF-IDF ---
tfidf_docs, _ = compute_tfidf(doc_tokens)

print(f"어휘 크기 : {len(vocab)}")
print(f"BoW 행렬  : {bow.shape}")
print(f"TF-IDF 행렬: {tfidf_docs.shape}\n")

def print_sim(mat, name):
    print(f"● {name} 코사인 유사도")
    print("      " + "".join(f"{d:>8s}" for d in doc_ids))
    for i, d in enumerate(doc_ids):
        row = "".join(f"{cos(mat[i], mat[j]):>8.3f}" for j in range(len(doc_ids)))
        print(f"{d:>6s}{row}")
    print()

print_sim(bow, 'BoW')
print_sim(tfidf_docs, 'TF-IDF')

# --- 어느 표현이 주제를 더 잘 나누는가? 말로 주장하지 말고 숫자로 확인한다 ---
# A1(샤라포바)·A2(부상 복귀)·A3(전자 판정) = 테니스(0),  A4·A5 = NLP(1)
TOPICS = [0, 0, 0, 1, 1]

def topic_separation(mat):
    """같은 주제 쌍 / 다른 주제 쌍의 평균 코사인 유사도를 돌려준다"""
    same, diff = [], []
    for i in range(len(TOPICS)):
        for j in range(i + 1, len(TOPICS)):
            (same if TOPICS[i] == TOPICS[j] else diff).append(cos(mat[i], mat[j]))
    return float(np.mean(same)), float(np.mean(diff))

s_bow,   d_bow   = topic_separation(bow)
s_tfidf, d_tfidf = topic_separation(tfidf_docs)
gap_bow,   gap_tfidf   = s_bow - d_bow,        s_tfidf - d_tfidf
ratio_bow, ratio_tfidf = s_bow / max(d_bow, 1e-12), s_tfidf / max(d_tfidf, 1e-12)

print("● 주제 구분 능력 (기사 쌍 10개를 같은 주제 / 다른 주제로 나눠 평균)")
print(f"{'표현':<8}{'같은 주제':>12}{'다른 주제':>12}{'격차':>10}{'비율':>10}")
print(f"{'BoW':<8}{s_bow:>12.4f}{d_bow:>12.4f}{gap_bow:>10.4f}{ratio_bow:>10.2f}")
print(f"{'TF-IDF':<8}{s_tfidf:>12.4f}{d_tfidf:>12.4f}{gap_tfidf:>10.4f}{ratio_tfidf:>10.2f}")

print("\n[관찰된 사실]")
print("→ 두 표현 모두 '같은 주제' 쌍의 평균 유사도가 '다른 주제' 쌍보다 높다. 즉 주제 구분 자체는 둘 다 된다.")
print(f"→ 절대 격차가 더 큰 쪽 : {'BoW' if gap_bow > gap_tfidf else 'TF-IDF'}"
      f"  ({max(gap_bow, gap_tfidf):.4f} vs {min(gap_bow, gap_tfidf):.4f})")
print(f"→ 상대 비율(같은 주제 ÷ 다른 주제)이 더 큰 쪽 : "
      f"{'BoW' if ratio_bow > ratio_tfidf else 'TF-IDF'}"
      f"  ({max(ratio_bow, ratio_tfidf):.2f} vs {min(ratio_bow, ratio_tfidf):.2f})")
print("→ TF-IDF 는 흔한 단어의 기여를 깎기 때문에 유사도 값 자체가 전반적으로 낮아진다.")
print("   여기서는 문서가 5편뿐이라 idf 추정이 매우 거칠고, 그래서 'TF-IDF 가 언제나 주제를 더")
print("   뚜렷하게 나눈다' 는 통념은 이 데이터에서 그대로 성립하지 않는다.")
print("   표현 방식의 우열은 이렇게 지표를 정해 직접 재 보아야 한다.")

In [ ]:
# 각 문서에서 TF-IDF 가 가장 높은 단어 = '그 문서다운' 단어
print("문서별 TF-IDF 상위 단어")
for i, d in enumerate(doc_ids):
    top = np.argsort(-tfidf_docs[i])[:8]
    print(f"  [{d}] " + ', '.join(vocab[j] for j in top))

## 3-3. Word2Vec 직접 구현 (Skip-gram + 네거티브 샘플링)

`gensim` 이 없어도 원리를 볼 수 있도록 NumPy 로 직접 구현합니다.

**학습 목표** : 중심 단어 $c$ 와 주변 단어 $o$ 에 대해
$\sigma(v_o \cdot v_c)$ 는 1에 가깝게, 무작위로 뽑은 오답 $k$ 개에 대해서는 0에 가깝게.

$$L = -\log\sigma(v_o^\top v_c) - \sum_{i=1}^{k}\log\sigma(-v_{n_i}^\top v_c)$$

In [ ]:
class SimpleWord2Vec:
    """Skip-gram with Negative Sampling (교육용 최소 구현)"""

    def __init__(self, dim=64, window=3, negative=5, min_count=2,
                 epochs=30, lr=0.05, seed=42):
        self.dim, self.window, self.negative = dim, window, negative
        self.min_count, self.epochs, self.lr = min_count, epochs, lr
        self.rng = np.random.default_rng(seed)

    def build_vocab(self, sentences):
        counts = Counter(w for s in sentences for w in s)
        self.vocab = [w for w, c in counts.most_common() if c >= self.min_count]
        self.widx = {w: i for i, w in enumerate(self.vocab)}
        freq = np.array([counts[w] for w in self.vocab], dtype=float)
        # 네거티브 샘플링 분포: 빈도^0.75 (Mikolov et al.)
        self.neg_dist = freq ** 0.75
        self.neg_dist /= self.neg_dist.sum()
        V = len(self.vocab)
        self.W  = (self.rng.random((V, self.dim)) - 0.5) / self.dim   # 중심 벡터
        self.C  = np.zeros((V, self.dim))                             # 문맥 벡터
        return V

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -6, 6)))

    def train(self, sentences, verbose=True):
        V = self.build_vocab(sentences)
        data = [[self.widx[w] for w in s if w in self.widx] for s in sentences]
        for ep in range(self.epochs):
            loss, n = 0.0, 0
            lr = self.lr * (1 - ep / self.epochs) + 1e-4   # 선형 감쇠
            for sent in data:
                for pos, center in enumerate(sent):
                    lo = max(0, pos - self.window)
                    hi = min(len(sent), pos + self.window + 1)
                    for ctx_pos in range(lo, hi):
                        if ctx_pos == pos:
                            continue
                        ctx = sent[ctx_pos]
                        negs = self.rng.choice(V, self.negative, p=self.neg_dist)
                        targets = np.concatenate(([ctx], negs))
                        labels = np.zeros(1 + self.negative); labels[0] = 1.0

                        vc = self.W[center]
                        uo = self.C[targets]
                        score = self._sigmoid(uo @ vc)
                        err = score - labels

                        self.C[targets] -= lr * np.outer(err, vc)
                        self.W[center]  -= lr * (err @ uo)

                        loss += -np.log(max(score[0], 1e-9)) \
                                - np.log(np.maximum(1 - score[1:], 1e-9)).sum()
                        n += 1
            if verbose and (ep + 1) % 10 == 0:
                print(f"  epoch {ep+1:3d}/{self.epochs}  평균 손실 {loss/max(n,1):.4f}")
        return self

    def __getitem__(self, word):
        return self.W[self.widx[word]]

    def most_similar(self, word, topn=8, positive=None, negative=None):
        norms = np.linalg.norm(self.W, axis=1, keepdims=True); norms[norms == 0] = 1e-9
        U = self.W / norms
        if positive or negative:
            q = np.zeros(self.dim)
            for w in (positive or []):
                q += self[w]
            for w in (negative or []):
                q -= self[w]
            exclude = set((positive or []) + (negative or []))
        else:
            q, exclude = self[word], {word}
        q = q / (np.linalg.norm(q) + 1e-9)
        sims = U @ q
        order = np.argsort(-sims)
        out = [(self.vocab[i], float(sims[i])) for i in order
               if self.vocab[i] not in exclude][:topn]
        return out


# 학습용 문장: 기사 5편의 문장을 전처리(불용어 유지 - 문맥 정보가 필요하므로 최소만 제거)
w2v_sentences = []
for a in ARTICLES:
    for s in split_sentences(a['text']):
        toks = [w for w in tokenize(s) if len(w) > 1]
        if len(toks) >= 4:
            w2v_sentences.append(toks)

print(f"학습 문장 수 : {len(w2v_sentences)}")
print("학습 시작 (수십 초 소요)")
w2v = SimpleWord2Vec(dim=64, window=3, negative=5, min_count=2, epochs=30).train(w2v_sentences)
print(f"학습 완료. 어휘 크기 = {len(w2v.vocab)}")

In [ ]:
# 학습된 임베딩이 의미를 담고 있는지, 몇 개 단어의 최근접 이웃을 뽑아 확인한다
#
# 주의 : 코퍼스가 작으면 the/was/or/with 같은 기능어가 거의 모든 단어와 함께 등장하기
#        때문에 최근접 이웃 상위가 전부 기능어로 채워져 버린다. 기능어를 걸러 내고
#        '내용어' 만 보아야 임베딩이 무엇을 배웠는지 판단할 수 있다.
FUNCTION_WORDS = STOPWORDS | {
    'therefore', 'however', 'also', 'now', 'then', 'one', 'two', 'many', 'much',
    'even', 'still', 'around', 'several', 'already', 'usually', 'finally',
    'basically', 'really', 'often', 'just', 'well', 'back', 'away', 'long'}


def content_neighbors(model, word, topn=6, pool=60):
    """최근접 이웃 중 기능어와 두 글자 이하 단어를 제외하고 내용어만 topn개 반환"""
    cand = model.most_similar(word, topn=pool)
    return [(w, s) for w, s in cand if w not in FUNCTION_WORDS and len(w) > 2][:topn]


for query in ['tennis', 'summary', 'vectors', 'players']:
    # min_count 미만으로 등장한 단어는 어휘에서 빠지므로 조회 전에 존재 여부를 확인한다
    if query in w2v.widx:
        print(f"'{query}' 와 가까운 단어 (기능어 제외)")
        for w, s in content_neighbors(w2v, query):
            print(f"    {w:<16s} {s:.3f}")
        print()
    else:
        print(f"'{query}' 는 어휘에 없습니다 (min_count 미달)\n")

print("→ 기능어를 걸러 내면 tennis-tour-court-match, vectors-words-word 처럼")
print("  같은 분야의 단어끼리 모이는 것이 보인다. 걸러 내기 전에는 was/or/with 가 상위를 차지했다.")

> **주의** — 학습 코퍼스가 매우 작으므로(수백 문장) 결과가 완벽하지 않습니다.
> 실제 Word2Vec은 수십억 단어로 학습합니다. 여기서는 *원리와 학습 과정*을 보는 것이 목적입니다.

In [ ]:
# gensim 이 있으면 같은 코퍼스로 학습해 비교 ([0-1] 설치 셀을 실행했으면 여기가 동작한다)
if gensim is not None:
    from gensim.models import Word2Vec
    g = Word2Vec(w2v_sentences, vector_size=64, window=3, min_count=2,
                 sg=1, negative=5, epochs=30, seed=42, workers=1)
    print("gensim Word2Vec 결과 (직접 구현과 같은 기준으로 기능어 제외)")
    for q in ['tennis', 'summary']:
        if q in g.wv:
            cand = [w for w, _ in g.wv.most_similar(q, topn=40)
                    if w not in FUNCTION_WORDS and len(w) > 2][:5]
            print(f"  '{q}' :", cand)
        else:
            print(f"  '{q}' 는 gensim 어휘에도 없습니다 (min_count 미달)")
    print("\n→ 직접 구현과 순위가 똑같지는 않다. 초기화·네거티브 샘플링 난수와")
    print("  학습률 스케줄이 다르기 때문이며, 작은 코퍼스일수록 이 차이가 크게 보인다.")
else:
    print("gensim 이 없어 비교를 건너뜁니다. ([0-1] 설치 셀을 실행했는지 확인하세요)")

## 3-4. 벡터 연산 — 유사도와 유추

$$vec(king) - vec(man) + vec(woman) \approx vec(queen)$$

작은 코퍼스에서는 이런 관계가 잘 나타나지 않으므로,
여기서는 **관계가 학습되는지 확인하는 방법 자체**를 익힙니다.

In [ ]:
# 유추(analogy) 과제를 벡터 덧셈·뺄셈으로 푼다
def analogy(model, a, b, c, topn=5):
    """a : b = c : ?   ->  vec(b) - vec(a) + vec(c)"""
    # 세 단어가 모두 어휘에 있어야 질의 벡터를 만들 수 있다
    missing = [w for w in (a, b, c) if w not in model.widx]
    if missing:
        print(f"어휘에 없는 단어: {missing}"); return []
    # positive/negative 로 넘기면 vec(b)+vec(c)-vec(a) 를 질의 벡터로 삼아 최근접 단어를 찾는다
    res = model.most_similar(None, topn=topn, positive=[b, c], negative=[a])
    print(f"{a} : {b}  =  {c} : ?")
    for w, s in res:
        print(f"    {w:<16s} {s:.3f}")
    return res

# 앞은 단수-복수라는 문법 관계, 뒤는 분야-대상이라는 의미 관계를 학습했는지 보는 시험이다
analogy(w2v, 'player', 'players', 'word')
print()
analogy(w2v, 'tennis', 'court', 'text')

In [ ]:
# 단어 벡터를 2차원으로 투영해 보기 (PCA, NumPy 로 직접)
def pca_2d(X):
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

show_words = [w for w in ['tennis', 'court', 'player', 'players', 'match', 'injury',
                          'summary', 'summaries', 'text', 'sentences', 'word', 'words',
                          'vectors', 'meaning', 'readers', 'article']
              if w in w2v.widx]
X = np.array([w2v[w] for w in show_words])
P = pca_2d(X)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(P[:, 0], P[:, 1], s=40)
for (x, y), w in zip(P, show_words):
    ax.annotate(w, (x, y), fontsize=10, xytext=(4, 4), textcoords='offset points')
ax.set_title('Word vectors projected to 2D (PCA)')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3-5. 문서 벡터와 활용

문서 벡터를 만드는 가장 간단한 방법은 **단어 벡터의 평균**입니다.
(TextRank에서 문장 벡터를 만든 방식과 같습니다.)

In [ ]:
# 문서에 나온 단어 벡터의 평균을 문서 벡터로 삼아 기사끼리의 유사도를 비교한다
def document_vector(model, text):
    words = [w for w in tokenize(text) if w in model.widx]   # 어휘에 없는 단어는 평균에서 제외한다
    if not words:
        return np.zeros(model.dim)        # 남는 단어가 하나도 없으면 영벡터로 처리한다
    return np.mean([model[w] for w in words], axis=0)

doc_vecs = np.array([document_vector(w2v, a['text']) for a in ARTICLES])

print("● 임베딩 평균 문서 벡터의 코사인 유사도")
print("      " + "".join(f"{d:>8s}" for d in doc_ids))
for i, d in enumerate(doc_ids):
    print(f"{d:>6s}" + "".join(f"{cos(doc_vecs[i], doc_vecs[j]):>8.3f}"
                               for j in range(len(doc_ids))))

print("\n● 가장 비슷한 기사 쌍 찾기")
# 모든 기사 쌍을 전수 비교한다. j 를 i+1 부터 돌려 자기 자신과 중복 쌍을 걸러 낸다
best = None
for i in range(len(doc_ids)):
    for j in range(i+1, len(doc_ids)):
        s = cos(doc_vecs[i], doc_vecs[j])
        if best is None or s > best[0]:
            best = (s, doc_ids[i], doc_ids[j])
print(f"  {best[1]} 와 {best[2]} (유사도 {best[0]:.3f})")

In [ ]:
# 문서 벡터로 간단한 주제 분류 (테니스=0, NLP=1)
labels = np.array([0, 0, 0, 1, 1])

def nearest_centroid_loo(vectors, labels):
    """Leave-One-Out 최근접 중심 분류 정확도"""
    correct = 0
    for i in range(len(vectors)):
        mask = np.ones(len(vectors), bool); mask[i] = False
        preds = {}
        for c in np.unique(labels):
            sel = mask & (labels == c)
            preds[c] = vectors[sel].mean(axis=0)
        pred = max(preds, key=lambda c: cos(vectors[i], preds[c]))
        correct += (pred == labels[i])
        print(f"  {doc_ids[i]}: 정답={labels[i]}  예측={pred}  {'O' if pred==labels[i] else 'X'}")
    return correct / len(vectors)

print("● TF-IDF 문서 벡터")
acc_tfidf = nearest_centroid_loo(tfidf_docs, labels)
print(f"  정확도 {acc_tfidf:.1%}\n")

print("● Word2Vec 평균 문서 벡터")
acc_w2v = nearest_centroid_loo(doc_vecs, labels)
print(f"  정확도 {acc_w2v:.1%}")

---
# 과제

아래 4개 과제를 **이 노트북 안에서** 수행하고, 실행 결과가 포함된 `.ipynb` 로 제출하세요.

- 파일명 : `학번_이름_6주차.ipynb`
- 코드뿐 아니라 **관찰과 해석을 글로** 작성해야 점수를 받습니다.
- 과제 1~3의 코드 셀 아래에는 **채점 셀** 이 있습니다. `통과!` 가 출력되어야 정답으로 인정됩니다.
  (과제 4는 서술형이므로 채점 셀이 없습니다.)

## 과제 1 — TextRank 하이퍼파라미터 실험 (30점)

`textrank_summarize()` 의 damping factor `d` 와 요약 문장 수 `n` 을 바꿔가며
ROUGE-1 / ROUGE-2 / ROUGE-L 의 평균 F1이 어떻게 달라지는지 표로 정리하세요.

- `d` ∈ {0.5, 0.85, 0.99}
- `n` ∈ {2, 3, 5}
- 기사 5편의 **평균** 점수를 사용

**구현할 것**

`experiment_textrank(d_values, n_values)` 가 9개 조합에 대해
`{'d': …, 'n': …, 'R1': …, 'R2': …, 'RL': …}` 딕셔너리 9개를 담은 **리스트**를 반환하도록 작성한다.
(`R1/R2/RL` 은 기사 5편에 대한 **평균 F1**)

**답안에 포함할 것**
1. 9개 조합의 결과 표
2. 가장 좋은 조합과 그 이유에 대한 설명 (3~5문장)
3. `n` 이 커질수록 recall 과 precision 이 각각 어떻게 변하는지, 왜 그런지 설명

In [ ]:
# ── 과제 1 ────────────────────────────────────────────────────────
def experiment_textrank(d_values=(0.5, 0.85, 0.99), n_values=(2, 3, 5)):
    """d × n 의 9개 조합 각각에 대해 기사 5편의 평균 ROUGE F1 을 계산하여
       [{'d':…, 'n':…, 'R1':…, 'R2':…, 'RL':…}, …]  (원소 9개) 를 반환한다."""
    # TODO 1 : d_values 와 n_values 의 모든 조합(9가지)을 순회한다.
    # TODO 2 : 각 조합에서 ARTICLES 5편에 textrank_summarize(a['text'], n=n, d=d) 를 적용한다.
    #          (반환값이 (요약문, 인덱스) 튜플이므로 [0] 으로 요약문만 꺼낸다)
    # TODO 3 : rouge_all(a['reference'], 요약문) 의 'ROUGE-1'/'ROUGE-2'/'ROUGE-L' f1 을 모아
    #          기사 5편에 대한 평균을 구한다.
    # TODO 4 : 조합마다 {'d','n','R1','R2','RL'} 딕셔너리를 만들어 리스트에 담아 반환한다.
    results = []
    for d in d_values:
        for n in n_values:
            r1, r2, rl, r1_p, r1_r = [], [], [], [], []
            for a in ARTICLES:
                cand = textrank_summarize(a['text'], n=n, d=d)[0]
                sc = rouge_all(a['reference'], cand)
                r1.append(sc['ROUGE-1']['f1'])
                r2.append(sc['ROUGE-2']['f1'])
                rl.append(sc['ROUGE-L']['f1'])
                # 서술 3번(n 과 recall/precision 의 관계)을 위해 ROUGE-1 P/R 도 함께 기록
                r1_p.append(sc['ROUGE-1']['precision'])
                r1_r.append(sc['ROUGE-1']['recall'])
            results.append({'d': d, 'n': n,
                            'R1': float(np.mean(r1)), 'R2': float(np.mean(r2)),
                            'RL': float(np.mean(rl)),
                            'R1_P': float(np.mean(r1_p)), 'R1_R': float(np.mean(r1_r))})
    return results


try:
    hw1_results = experiment_textrank()
    import pandas as pd
    hw1_df = pd.DataFrame(hw1_results).round(4)
    print("● d × n 조합별 기사 5편 평균 ROUGE F1")
    display(hw1_df)
    _best = max(hw1_results, key=lambda r: r['R1'])
    print(f"R1 기준 최고 조합 : d={_best['d']}, n={_best['n']}  (R1={_best['R1']:.4f})")
except NotImplementedError as e:
    hw1_results = []          # 아래 셀이 참조해도 오류가 나지 않도록 안전한 기본값
    print('▶ 미구현 :', e)

In [ ]:
# [과제 1] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    _res = experiment_textrank()
    assert isinstance(_res, list) and len(_res) == 9, \
        f'9개 조합의 결과가 필요합니다. (현재 {len(_res) if hasattr(_res, "__len__") else "?"}개)'
    assert all(isinstance(r, dict) for r in _res), '각 원소는 딕셔너리여야 합니다.'
    _need = {'d', 'n', 'R1', 'R2', 'RL'}
    assert all(_need <= set(r) for r in _res), f'각 딕셔너리는 {sorted(_need)} 키를 모두 가져야 합니다.'
    assert {(r['d'], r['n']) for r in _res} == {(d, n) for d in (0.5, 0.85, 0.99) for n in (2, 3, 5)}, \
        'd ∈ {0.5, 0.85, 0.99} 와 n ∈ {2, 3, 5} 의 9개 조합이 모두 있어야 합니다.'
    assert all(isinstance(r[k], (int, float)) and not isinstance(r[k], bool) and 0.0 < r[k] < 1.0
               for r in _res for k in ('R1', 'R2', 'RL')), \
        'R1/R2/RL 은 0과 1 사이의 실수여야 합니다. (None 이나 자리표시자를 그대로 두면 안 됩니다)'

    _r1 = {round(r['R1'], 6) for r in _res}
    assert len(_r1) >= 3, \
        f'9개 조합의 R1 이 사실상 같은 값입니다({len(_r1)}종). 조합마다 실제로 다시 계산해야 합니다.'

    _by = {(r['d'], r['n']): r for r in _res}
    assert _by[(0.85, 5)]['R1'] > _by[(0.85, 2)]['R1'], \
        'n 을 2에서 5로 늘리면 평균 R1 이 올라가야 합니다. 계산 과정을 확인하세요.'
    assert abs(_by[(0.85, 3)]['R1'] - 0.4043) < 0.03, \
        f"d=0.85, n=3 의 평균 R1 이 기대 범위를 벗어났습니다. (현재 {_by[(0.85, 3)]['R1']:.4f})"
    assert abs(_by[(0.5, 5)]['R2'] - 0.1912) < 0.03, \
        f"d=0.5, n=5 의 평균 R2 가 기대 범위를 벗어났습니다. (현재 {_by[(0.5, 5)]['R2']:.4f})"
    assert abs(_by[(0.99, 2)]['RL'] - 0.2866) < 0.03, \
        f"d=0.99, n=2 의 평균 ROUGE-L 이 기대 범위를 벗어났습니다. (현재 {_by[(0.99, 2)]['RL']:.4f})"

    print(f'통과!  9개 조합 계산 완료 — R1 범위 {min(_r1):.4f} ~ {max(_r1):.4f}')
except NotImplementedError as e:
    print('▶ 미구현 :', e)

### 과제 1 답안 (서술)

*(여기에 관찰과 해석을 작성하세요)*

1. 가장 좋은 조합 :
2. 그 이유 :
3. `n` 과 recall / precision 의 관계 :

## 과제 2 — 마르코프 체인 생성기 개선 (30점)

1. `build_ngram_model()` 로 **bigram(n=2)**, **trigram(n=3)**, **4-gram** 모델을 만들고
   각각 **같은 seed** 로 3개씩 생성하세요.
   - 이 모델들의 상태는 튜플이므로 반드시 `generate_ngram(model, n=n, ...)` 으로 생성합니다.
     (`generate_text()` 는 `build_bigram_model()` 이 만든 **문자열 상태** 전용입니다)
2. 세 결과의 **자연스러움**을 직접 읽고 비교해 서술하세요.
3. 각 모델에서 "후보가 1개뿐인 상태"의 비율을 계산하고,
   n이 커질수록 왜 원문을 그대로 외우게 되는지 설명하세요.
4. (심화) `temperature` 와 `top_k` 를 조합해 **반복이 가장 적으면서도 읽을 만한**
   설정을 찾고, 그 근거를 제시하세요.
   - 지표 : 생성된 텍스트의 고유 단어 비율 `len(set(tokens))/len(tokens)` (아래 `diversity()`)

**구현할 것**

| 함수 | 반환값 |
|---|---|
| `compare_ngram_generation(ns, n_words, seeds)` | `{n: [(생성문, 고유단어비율), ...3개], ...}` |
| `single_candidate_ratio(ns)` | `{n: 후보가 1개뿐인 상태의 비율}` |
| `find_best_setting(...)` | `{'temperature': …, 'top_k': …, 'diversity': …}` |

In [ ]:
# ── 과제 2 ────────────────────────────────────────────────────────
def diversity(text):
    """생성 텍스트의 고유 단어 비율 (1에 가까울수록 반복이 적음)"""
    toks = tokenize(text)
    return len(set(toks)) / max(len(toks), 1)


# 참고 예시 : build_ngram_model() 의 상태는 튜플이므로 generate_ngram() 으로 생성한다.
#            (generate_text() 에 넣으면 "generate_ngram 을 쓰라"는 TypeError 가 난다)
demo_model = build_ngram_model(GEN_TOKENS, n=2)
demo_text = generate_ngram(demo_model, n=2, n_words=60, temperature=1.0, seed=1)
print(demo_text)
print(f"\n고유 단어 비율 = {diversity(demo_text):.3f}\n")


def compare_ngram_generation(ns=(2, 3, 4), n_words=60, seeds=(1, 2, 3)):
    """n 별로 seeds 개수만큼 텍스트를 생성해 {n: [(생성문, 고유단어비율), ...]} 를 반환한다."""
    # TODO 1 : n 마다 build_ngram_model(GEN_TOKENS, n=n) 로 모델을 만든다.
    # TODO 2 : 세 모델 모두 같은 seeds 를 써서 generate_ngram(model, n=n, n_words=n_words, seed=s) 로 생성한다.
    # TODO 3 : (생성문, diversity(생성문)) 튜플을 모아 n 을 키로 하는 딕셔너리로 반환한다.
    result = {}
    for n in ns:
        model = build_ngram_model(GEN_TOKENS, n=n)
        result[n] = []
        for s in seeds:
            text = generate_ngram(model, n=n, n_words=n_words, seed=s)
            result[n].append((text, diversity(text)))
    return result


def single_candidate_ratio(ns=(2, 3, 4)):
    """n 별로 '다음 단어 후보가 1개뿐인 상태' 의 비율을 {n: 비율} 로 반환한다."""
    # TODO 4 : build_ngram_model 로 만든 모델의 값(Counter) 중 len(counter) == 1 인 상태를 세고
    #          전체 상태 수로 나눈다.
    ratios = {}
    for n in ns:
        model = build_ngram_model(GEN_TOKENS, n=n)
        single = sum(1 for counter in model.values() if len(counter) == 1)
        ratios[n] = single / len(model)
    return ratios


def diversity_grid(n=2, temperatures=(0.6, 0.8, 1.0, 1.3),
                   top_ks=(None, 3, 5, 10), seeds=(1, 2, 3), n_words=60):
    """temperature × top_k 조합마다 seeds 로 생성한 텍스트의 평균 고유 단어 비율"""
    model = build_ngram_model(GEN_TOKENS, n=n)
    rows = []
    for T in temperatures:
        for k in top_ks:
            dvs = [diversity(generate_ngram(model, n=n, n_words=n_words,
                                            temperature=T, top_k=k, seed=s))
                   for s in seeds]
            rows.append({'temperature': T, 'top_k': k, 'diversity': float(np.mean(dvs))})
    return rows


def find_best_setting(n=2, temperatures=(0.6, 0.8, 1.0, 1.3),
                      top_ks=(None, 3, 5, 10), seeds=(1, 2, 3)):
    """반복이 가장 적은(고유 단어 비율이 가장 높은) 디코딩 설정을 찾아
       {'temperature': …, 'top_k': …, 'diversity': …} 를 반환한다."""
    # TODO 5 : temperature × top_k 격자를 순회하며, seeds 로 여러 번 생성해 diversity 평균을 구한다.
    # TODO 6 : 평균 diversity 가 가장 높은 설정을 위 세 키를 가진 딕셔너리로 반환한다.
    rows = diversity_grid(n, temperatures, top_ks, seeds)
    return max(rows, key=lambda r: r['diversity'])


try:
    hw2_gen = compare_ngram_generation()
    for n in sorted(hw2_gen):
        print(f"▶ {n}-gram")
        for t, dv in hw2_gen[n]:
            print(f"   (고유 단어 비율 {dv:.3f}) {t[:88]} ...")
        print()
    print("● 후보가 1개뿐인 상태의 비율 :",
          {n: f"{r:.1%}" for n, r in single_candidate_ratio().items()})
    print("● 반복이 가장 적은 설정 :", find_best_setting())
except NotImplementedError as e:
    hw2_gen = {}              # 아래 셀이 참조해도 오류가 나지 않도록 안전한 기본값
    print('▶ 미구현 :', e)

In [ ]:
# [과제 2] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    _gen = compare_ngram_generation()
    assert isinstance(_gen, dict) and set(_gen) == {2, 3, 4}, 'n = 2, 3, 4 를 키로 갖는 딕셔너리여야 합니다.'
    for _n in (2, 3, 4):
        _items = _gen[_n]
        assert isinstance(_items, list) and len(_items) == 3, f'{_n}-gram 결과가 3개여야 합니다.'
        assert all(isinstance(x, tuple) and len(x) == 2 for x in _items), '(생성문, 고유단어비율) 튜플이어야 합니다.'
        assert all(isinstance(t, str) and len(tokenize(t)) >= 30 for t, _ in _items), \
            f'{_n}-gram 의 각 생성문은 30단어 이상의 문자열이어야 합니다.'
        assert len({t for t, _ in _items}) == 3, \
            f'{_n}-gram 의 세 결과가 모두 같습니다. seed 를 서로 다르게 주어 생성해야 합니다.'
        assert all(abs(dv - diversity(t)) < 1e-9 for t, dv in _items), \
            f'{_n}-gram 의 두 번째 원소는 그 생성문의 diversity() 값이어야 합니다. (고정값 금지)'

    _ratio = single_candidate_ratio()
    assert isinstance(_ratio, dict) and set(_ratio) == {2, 3, 4}, 'n = 2, 3, 4 를 키로 갖는 딕셔너리여야 합니다.'
    assert all(isinstance(v, (int, float)) and 0.0 < v <= 1.0 for v in _ratio.values()), \
        '비율은 0과 1 사이의 실수여야 합니다.'
    assert _ratio[2] < _ratio[3] < _ratio[4], \
        f'n 이 커질수록 후보가 1개뿐인 상태의 비율이 커져야 합니다. (현재 {[round(_ratio[k], 3) for k in (2, 3, 4)]})'
    assert abs(_ratio[2] - 0.7424) < 0.02, f'n=2 의 비율이 기대값과 다릅니다. (현재 {_ratio[2]:.4f})'
    assert abs(_ratio[4] - 0.9923) < 0.02, f'n=4 의 비율이 기대값과 다릅니다. (현재 {_ratio[4]:.4f})'

    _best = find_best_setting()
    assert isinstance(_best, dict) and {'temperature', 'top_k', 'diversity'} <= set(_best), \
        "{'temperature', 'top_k', 'diversity'} 키를 가진 딕셔너리를 반환해야 합니다."
    assert isinstance(_best['temperature'], (int, float)) and 0 < _best['temperature'] <= 2.0, \
        'temperature 는 0보다 크고 2 이하인 값이어야 합니다.'
    assert _best['top_k'] is None or (isinstance(_best['top_k'], int) and _best['top_k'] >= 2), \
        'top_k 는 None 이거나 2 이상의 정수여야 합니다.'
    assert isinstance(_best['diversity'], (int, float)) and 0.4 < _best['diversity'] <= 1.0, \
        f"고유 단어 비율이 {_best['diversity']:.3f} 입니다. greedy 디코딩(약 0.10)보다 뚜렷하게 높아야 합니다."

    print(f"통과!  n별 단일후보 비율 {[f'{_ratio[k]:.1%}' for k in (2, 3, 4)]}, "
          f"최적 설정 T={_best['temperature']} / top_k={_best['top_k']} "
          f"(고유 단어 비율 {_best['diversity']:.3f})")
except NotImplementedError as e:
    print('▶ 미구현 :', e)

In [ ]:
# [과제 2 보조] temperature × top_k 전체 격자 — 최고값 하나만 보지 않고 경향을 함께 본다
_rows = diversity_grid()
grid = pd.DataFrame({'temperature': [r['temperature'] for r in _rows],
                     'top_k': [str(r['top_k']) for r in _rows],     # None 이 NaN 으로 바뀌지 않도록 문자열로
                     'diversity': [r['diversity'] for r in _rows]})
print("● bigram, seed 1~3 평균 고유 단어 비율")
print(grid.pivot(index='temperature', columns='top_k', values='diversity')
          [['3', '5', '10', 'None']].round(3).to_string())

_m2 = build_ngram_model(GEN_TOKENS, n=2)
for T, k in [(0.6, 3), (1.0, 5), (1.3, None)]:
    t = generate_ngram(_m2, n=2, n_words=40, temperature=T, top_k=k, seed=1)
    print(f"\n▶ T={T}, top_k={k}  (고유 단어 비율 {diversity(t):.3f})\n  {t}")


### 과제 2 답안 (서술)

*(여기에 비교와 해석을 작성하세요)*

1. bigram / trigram / 4-gram 결과 비교 :
2. 후보가 1개뿐인 상태 비율과 그 의미 :
3. 가장 좋은 temperature / top_k 조합과 근거 :

## 과제 3 — 단어 임베딩 분석 (30점)

`SimpleWord2Vec` 으로 학습한 벡터(`w2v`)를 사용해 다음을 수행하세요.

1. **의미가 가까운 단어 쌍 5개**를 찾아 유사도와 함께 제시하고, 왜 가깝게 학습됐는지 설명
   - 기능어(`FUNCTION_WORDS`)와 두 글자 이하의 단어는 제외한다.
     (걸러 내지 않으면 상위가 전부 `was/or/with` 같은 기능어로 채워진다)
2. **유추(analogy) 3개**를 시도하고 결과 제시 (성공/실패 무관)
3. **실패 사례 분석** : 기대와 다른 결과가 나온 경우를 최소 2개 골라
   원인을 아래 관점에서 분석
   - 코퍼스 크기 / 단어 빈도 (`min_count` 에 걸린 단어인가?)
   - 윈도우 크기 (`window`)
   - 다의어 문제 (정적 임베딩의 한계)
4. **하이퍼파라미터 실험** : `dim` 과 `window` 를 각각 2가지씩 바꿔 재학습하고
   유사 단어 결과가 어떻게 달라지는지 관찰

**구현할 것**

| 함수 | 반환값 |
|---|---|
| `similar_pairs(model, topk=5)` | `[(단어1, 단어2, 유사도), ...]` (유사도 내림차순) |
| `run_analogies(model, triples)` | `[(a, b, c, [(단어, 점수), ...]), ...]` |
| `hyperparam_experiment(sentences, dims, windows)` | `{(dim, window): [최근접 단어, ...]}` |

In [ ]:
# ── 과제 3 ────────────────────────────────────────────────────────
def similar_pairs(model, topk=5):
    """어휘 안의 '내용어' 쌍 가운데 코사인 유사도가 높은 상위 topk 쌍을
       [(단어1, 단어2, 유사도), ...] (유사도 내림차순) 으로 반환한다."""
    # TODO 1 : model.vocab 중 FUNCTION_WORDS 에 없고 길이가 3 이상인 단어만 후보로 남긴다.
    # TODO 2 : 서로 다른 두 단어의 모든 조합에 대해 cos(model[w1], model[w2]) 를 계산한다.
    #          (같은 쌍을 두 번 세지 않도록 itertools.combinations 를 쓰면 편하다)
    # TODO 3 : 유사도 내림차순으로 정렬해 상위 topk 쌍을 반환한다.
    cands = [w for w in model.vocab if w not in FUNCTION_WORDS and len(w) >= 3]
    pairs = [(w1, w2, cos(model[w1], model[w2]))
             for w1, w2 in itertools.combinations(cands, 2)]
    pairs.sort(key=lambda x: -x[2])
    return pairs[:topk]


def run_analogies(model, triples=(('player', 'players', 'word'),
                                  ('tennis', 'court', 'text'),
                                  ('word', 'words', 'player'))):
    """a : b = c : ? 유추를 세 번 수행해 [(a, b, c, [(단어, 점수), ...]), ...] 를 반환한다."""
    # TODO 4 : 세 단어가 모두 model.widx 에 있는지 먼저 확인한다. (없으면 그 유추는 건너뛴다)
    # TODO 5 : model.most_similar(None, topn=3, positive=[b, c], negative=[a]) 로 답 후보를 구한다.
    results = []
    for a, b, c in triples:
        if not all(w in model.widx for w in (a, b, c)):
            print(f"   어휘에 없는 단어가 있어 건너뜀 : {(a, b, c)}")
            continue
        res = model.most_similar(None, topn=3, positive=[b, c], negative=[a])
        results.append((a, b, c, res))
    return results


def hyperparam_experiment(sentences, dims=(16, 64), windows=(2, 5), query='tennis', epochs=15):
    """dim × window 조합마다 새로 학습해 query 의 최근접 단어 목록을
       {(dim, window): [단어, 단어, ...]} 로 반환한다."""
    # TODO 6 : SimpleWord2Vec(dim=…, window=…, epochs=epochs).train(sentences, verbose=False) 로 재학습
    # TODO 7 : 학습된 모델에서 query 의 최근접 단어 5개를 '단어만' 골라 목록으로 저장한다.
    results = {}
    for dim in dims:
        for window in windows:
            m = SimpleWord2Vec(dim=dim, window=window, epochs=epochs).train(sentences, verbose=False)
            # 기능어를 거르지 않으면 was/or/with 만 보이므로 content_neighbors 로 내용어만 본다
            results[(dim, window)] = [w for w, _ in content_neighbors(m, query, topn=5)]
    return results


try:
    print("● 의미가 가까운 내용어 쌍 상위 5")
    for w1, w2, s in similar_pairs(w2v):
        print(f"   {w1:<14s} {w2:<14s} {s:.3f}")

    print("\n● 유추 (a : b = c : ?)")
    for a_, b_, c_, res in run_analogies(w2v):
        print(f"   {a_} : {b_} = {c_} : ?  →  {[(w, round(s, 3)) for w, s in res]}")

    print("\n● 하이퍼파라미터 실험 (재학습에 십여 초 걸립니다)")
    for (dim_, win_), words in hyperparam_experiment(w2v_sentences).items():
        print(f"   dim={dim_:<4d} window={win_:<3d} → {words}")
except NotImplementedError as e:
    print('▶ 미구현 :', e)

In [ ]:
# [과제 3] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    _p = similar_pairs(w2v)
    assert isinstance(_p, list) and len(_p) == 5, '가까운 단어 쌍 5개를 반환해야 합니다.'
    assert all(isinstance(x, tuple) and len(x) == 3 for x in _p), '(단어1, 단어2, 유사도) 튜플이어야 합니다.'
    assert all(w1 in w2v.widx and w2 in w2v.widx for w1, w2, _ in _p), '어휘에 없는 단어가 들어 있습니다.'
    assert all(w1 != w2 for w1, w2, _ in _p), '같은 단어끼리 짝지을 수 없습니다.'
    assert len({frozenset((w1, w2)) for w1, w2, _ in _p}) == 5, '같은 쌍이 중복되었습니다.'
    assert not any(w in FUNCTION_WORDS or len(w) <= 2 for w1, w2, _ in _p for w in (w1, w2)), \
        '기능어(FUNCTION_WORDS)와 두 글자 이하 단어는 제외해야 합니다.'
    _sims = [s for _, _, s in _p]
    assert all(isinstance(s, (int, float)) and -1.001 <= s <= 1.001 for s in _sims), \
        '유사도는 -1 과 1 사이의 실수여야 합니다.'
    assert len({round(s, 6) for s in _sims}) >= 2, '모든 쌍의 유사도가 같습니다. 실제로 계산해야 합니다.'
    assert _sims == sorted(_sims, reverse=True), '유사도 내림차순으로 정렬해야 합니다.'
    assert max(_sims) > 0.5, f'가장 가까운 쌍의 유사도가 {max(_sims):.3f} 로 너무 낮습니다.'
    for w1, w2, s in _p:
        assert abs(s - cos(w2v[w1], w2v[w2])) < 1e-6, \
            f"({w1}, {w2}) 에 적힌 값이 실제 코사인 유사도와 다릅니다."

    _a = run_analogies(w2v)
    assert isinstance(_a, list) and len(_a) == 3, '유추 3개의 결과를 반환해야 합니다.'
    for _x in _a:
        assert isinstance(_x, tuple) and len(_x) == 4, '(a, b, c, 결과리스트) 형태여야 합니다.'
        _aa, _bb, _cc, _rr = _x
        assert all(w in w2v.widx for w in (_aa, _bb, _cc)), \
            f'어휘에 없는 단어로 유추를 시도했습니다 : {(_aa, _bb, _cc)}'
        assert isinstance(_rr, list) and _rr, f'{_aa}:{_bb}={_cc}:? 의 결과가 비어 있습니다.'
        assert all(isinstance(t, tuple) and len(t) == 2 for t in _rr), '(단어, 점수) 튜플 목록이어야 합니다.'
        assert all(w in w2v.widx for w, _ in _rr), '답 후보는 어휘 안의 단어여야 합니다.'
        assert not ({w for w, _ in _rr} & {_aa, _bb, _cc}), '질의에 쓴 단어 자신이 답에 들어가면 안 됩니다.'
    assert len({tuple(w for w, _ in _x[3]) for _x in _a}) >= 2, \
        '세 유추의 결과가 모두 같습니다. 실제로 계산했는지 확인하세요.'

    _h = hyperparam_experiment(w2v_sentences)
    assert isinstance(_h, dict) and len(_h) >= 4, 'dim × window 4개 조합의 결과가 필요합니다.'
    assert all(isinstance(k, tuple) and len(k) == 2 for k in _h), '키는 (dim, window) 튜플이어야 합니다.'
    assert all(isinstance(v, list) and len(v) >= 3 for v in _h.values()), \
        '조합마다 최근접 단어 3개 이상의 목록이 필요합니다.'
    assert all(isinstance(w, str) for v in _h.values() for w in v), '최근접 단어 목록은 문자열 목록이어야 합니다.'
    assert len({tuple(v) for v in _h.values()}) >= 2, \
        '모든 조합의 결과가 완전히 같습니다. 조합마다 실제로 재학습했는지 확인하세요.'

    print(f'통과!  가까운 쌍 최고 유사도 {max(_sims):.3f}, 유추 {len(_a)}개, '
          f'하이퍼파라미터 조합 {len(_h)}개')
except NotImplementedError as e:
    print('▶ 미구현 :', e)

In [ ]:
# [과제 3 보조] 실패 사례 분석용 — 유추·질의 단어들의 코퍼스 빈도와 어휘 포함 여부
word_counts = Counter(w for s in w2v_sentences for w in s)
check = ['player', 'players', 'word', 'words', 'tennis', 'court', 'text',
         'summary', 'summaries', 'king', 'queen', 'injury', 'match']
print(f"{'단어':<12}{'빈도':>6}   어휘 포함 (min_count=2)")
for w in check:
    print(f"{w:<12}{word_counts[w]:>6}   {'O' if w in w2v.widx else 'X'}")
print(f"\n전체 학습 토큰 수 : {sum(word_counts.values())},  어휘 크기 : {len(w2v.vocab)}")


### 과제 3 답안 (서술)

*(여기에 분석을 작성하세요)*

1. 가까운 단어 쌍 5개와 이유 :
2. 유추 3개 결과 :
3. 실패 사례 분석 :
4. 하이퍼파라미터 실험 관찰 :

## 과제 4 — 성찰 (10점)

**추출적 요약과 추상적(생성적) 요약 중 무엇을 언제 써야 하는가?**

다음 세 가지 상황을 각각 예로 들어, 어떤 방식을 선택할지와 그 이유를
**300자 내외**로 서술하세요.

- 의료 기록 요약
- 소셜 미디어 게시물 요약
- 사내 회의록 요약

평가 요소 : 두 방식의 장단점을 정확히 이해했는가, 상황과 근거를 구체적으로 연결했는가.

### 과제 4 답안

*(여기에 300자 내외로 작성하세요)*

---
## 마무리 체크리스트

- [ ] 모든 셀이 오류 없이 실행되었는가
- [ ] 과제 1~3의 **채점 셀이 모두 `통과!` 를 출력**했는가
- [ ] 과제 1~4의 코드와 **서술 답안**을 모두 작성했는가
- [ ] 파일명을 `학번_이름_6주차.ipynb` 로 바꿨는가
- [ ] 실행 결과가 저장된 상태로 제출하는가

### 자주 발생하는 문제

| 증상 | 해결 |
|---|---|
| `AttributeError: 'tuple' object has no attribute 'isalpha'` | `build_ngram_model()` 의 모델을 `generate_text()` 에 넣은 것. `generate_ngram(model, n=...)` 을 쓸 것 |
| 그래프에서 한글이 □ 로 보임 | `[0-1]` 설치 셀 실행 후 `[0-2]` 셀을 다시 실행 |
| `gensim 이 없어 비교를 건너뜁니다` | `[0-1]` 설치 셀을 실행하지 않은 것 |
| 생성 결과가 매번 달라짐 | `generate_ngram(..., seed=...)` 로 시드를 지정할 것 |

### 더 공부하고 싶다면

| 주제 | 키워드 |
|---|---|
| 추상 요약 | BART, T5, PEGASUS, KoBART |
| 요약 평가 | ROUGE, BERTScore, 사실성(faithfulness) 평가 |
| 생성 제어 | beam search, top-p, repetition penalty |
| 임베딩 | GloVe, FastText, Doc2Vec, Sentence-BERT |
| 문맥 임베딩 | ELMo, BERT, 다의어 해소 |